# Assignment 3: Product Sales Analysis using Apache Spark

### Aim
To write a Spark program to calculate the number of products sold in each category, using sample sales/order data.

### Description
Apache Spark is a distributed data processing engine widely used for handling large-scale datasets efficiently. In this experiment, sample order data (OrderID, Product, Category, QuantitySold) is loaded into a Spark DataFrame. Using Spark's DataFrame API, the data is grouped by product category, and aggregation functions are applied to calculate the total number of units sold and the number of distinct products under each category. The same aggregation is also performed using Spark's RDD API with a map and reduceByKey operation, to demonstrate the underlying map-reduce style computation that Spark DataFrames abstract away. This experiment highlights how Spark can be used to efficiently summarize and analyze large volumes of transactional/sales data.

## Tools Used
- Python  
- PySpark (SparkSession, DataFrame API, RDD API)

In [1]:
!pip install pyspark -q

In [2]:
# starting a spark session
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("ProductSalesByCategory").master("local[*]").getOrCreate()
print(f"Spark session started - version {spark.version}")

Spark session started - version 4.0.4


In [3]:
# sample order data - product, category, quantity sold per order
sales_data = [
    (101, "Notebook", "Stationery", 12),
    (102, "Pen Set", "Stationery", 25),
    (103, "Sneakers", "Footwear", 4),
    (104, "Sandals", "Footwear", 6),
    (105, "Bluetooth Speaker", "Electronics", 3),
    (106, "Earphones", "Electronics", 15),
    (107, "Wall Clock", "Home Decor", 5),
    (108, "Table Lamp", "Home Decor", 7),
    (109, "Notebook", "Stationery", 8),
    (110, "Earphones", "Electronics", 10),
    (111, "Sneakers", "Footwear", 2),
    (112, "Wall Clock", "Home Decor", 3),
]

columns = ["OrderID", "Product", "Category", "QuantitySold"]
sales_df = spark.createDataFrame(sales_data, schema=columns)
sales_df.show()

+-------+-----------------+-----------+------------+
|OrderID|          Product|   Category|QuantitySold|
+-------+-----------------+-----------+------------+
|    101|         Notebook| Stationery|          12|
|    102|          Pen Set| Stationery|          25|
|    103|         Sneakers|   Footwear|           4|
|    104|          Sandals|   Footwear|           6|
|    105|Bluetooth Speaker|Electronics|           3|
|    106|        Earphones|Electronics|          15|
|    107|       Wall Clock| Home Decor|           5|
|    108|       Table Lamp| Home Decor|           7|
|    109|         Notebook| Stationery|           8|
|    110|        Earphones|Electronics|          10|
|    111|         Sneakers|   Footwear|           2|
|    112|       Wall Clock| Home Decor|           3|
+-------+-----------------+-----------+------------+



In [4]:
# grouping by category and summing up quantity sold
category_totals = (sales_df.groupBy("Category")
                            .agg(F.sum("QuantitySold").alias("TotalUnitsSold"))
                            .orderBy(F.desc("TotalUnitsSold")))

category_totals.show()

+-----------+--------------+
|   Category|TotalUnitsSold|
+-----------+--------------+
| Stationery|            45|
|Electronics|            28|
| Home Decor|            15|
|   Footwear|            12|
+-----------+--------------+



In [5]:
# checking how many different products fall under each category
product_variety = (sales_df.groupBy("Category")
                            .agg(F.countDistinct("Product").alias("DistinctProducts")))

product_variety.show()

+-----------+----------------+
|   Category|DistinctProducts|
+-----------+----------------+
| Stationery|               2|
|Electronics|               2|
|   Footwear|               2|
| Home Decor|               2|
+-----------+----------------+



In [6]:
# checking how many different products fall under each category
product_variety = (sales_df.groupBy("Category")
                            .agg(F.countDistinct("Product").alias("DistinctProducts")))

product_variety.show()

+-----------+----------------+
|   Category|DistinctProducts|
+-----------+----------------+
| Stationery|               2|
|Electronics|               2|
|   Footwear|               2|
| Home Decor|               2|
+-----------+----------------+



In [7]:
# doing the same aggregation using RDDs, closer to the raw map-reduce style
category_rdd = sales_df.rdd.map(lambda row: (row["Category"], row["QuantitySold"]))
category_reduced = category_rdd.reduceByKey(lambda a, b: a + b)

for category, total in category_reduced.collect():
    print(f"{category}: {total} units sold")

Footwear: 12 units sold
Home Decor: 15 units sold
Stationery: 45 units sold
Electronics: 28 units sold


In [8]:
spark.stop()
print("Spark session stopped.")

Spark session stopped.
